In [20]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, random_split, DataLoader
import torch.optim as optim
import torchvision.transforms as transforms
from PIL import Image

In [21]:
DATA_DIR = Path("../../data")

In [22]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

In [23]:
class ImageDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        image = Image.open(file).convert("RGB")
        label = 0 if file.stem.startswith("cat.") else 1

        if self.transform:
            image = self.transform(image)

        return image, label

In [24]:
training_files = list(Path(f"{DATA_DIR}/train").glob("*"))

In [25]:
# 데이터 증강 선택: 쓰고 싶은 항목만 True
AUG = {
    "hflip": True,      # 좌우 반전
    "rotation": True,   # ±15도 회전
    "crop": True,       # 랜덤 크롭 후 리사이즈 (확대/위치 변화)
    "color": True,     # 밝기/대비/채도 변화
    "affine": True,    # 이동/기울이기
    "erasing": True,   # 일부 영역 지우기 (Random Erasing)
}

aug = []
if AUG["crop"]:
    aug.append(transforms.RandomResizedCrop(128, scale=(0.8, 1.0)))
else:
    aug.append(transforms.Resize((128, 128)))
if AUG["hflip"]:
    aug.append(transforms.RandomHorizontalFlip())
if AUG["rotation"]:
    aug.append(transforms.RandomRotation(15))
if AUG["color"]:
    aug.append(transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2))
if AUG["affine"]:
    aug.append(transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), shear=10))
aug.append(transforms.ToTensor())
if AUG["erasing"]:
    aug.append(transforms.RandomErasing(p=0.5))  # ToTensor 뒤에 와야 함

train_transform = transforms.Compose(aug)

# 검증/테스트는 증강 없이
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor()
])

In [26]:
# random_split은 transform을 공유하므로 파일 목록을 먼저 나눈다
train_size = int(len(training_files) * 0.8)
val_size = len(training_files) - train_size

train_files, val_files = random_split(
    training_files,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

In [27]:
train_dataset = ImageDataset(list(train_files), transform=train_transform)
val_dataset = ImageDataset(list(val_files), transform=transform)

print("Train:", len(train_dataset))
print("Validation:", len(val_dataset))

Train: 20000
Validation: 5000


In [28]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

In [ ]:
import torch
import torch.nn as nn

class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 16 * 16, 256),
            nn.ReLU(),
            nn.Linear(256, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [30]:
model = CNN().to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

In [31]:
epochs = 200

history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_loss = float("inf")

In [32]:
for epoch in range(epochs):

    # Train
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # Validation
    model.eval()

In [33]:
val_loss = 0
val_correct = 0
val_total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        val_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        val_correct += (predictions == labels).sum().item()
        val_total += labels.size(0)

val_loss /= val_total
val_acc = val_correct / val_total

# History
history["train_loss"].append(train_loss)
history["train_acc"].append(train_acc)
history["val_loss"].append(val_loss)
history["val_acc"].append(val_acc)

# Best Model 저장
if val_loss < best_val_loss:
    best_val_loss = val_loss
    torch.save(model.state_dict(), "best_model.pth")

print(
    f"Epoch [{epoch+1}/{epochs}] "
    f"Train Loss: {train_loss:.4f} "
    f"Train Acc: {train_acc:.4f} | "
    f"Val Loss: {val_loss:.4f} "
    f"Val Acc: {val_acc:.4f}"
)

Epoch [200/200] Train Loss: 0.1514 Train Acc: 0.9393 | Val Loss: 0.1514 Val Acc: 0.9458


In [34]:
test_files = sorted(Path(f"{DATA_DIR}/test").glob("*.jpg"))

In [35]:
class TestDataset(Dataset):
    def __init__(self, files, transform=None):
        self.files = files
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file = self.files[idx]

        image = Image.open(file).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, file.name

In [36]:
test_dataset = TestDataset(
    test_files,
    transform=transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [43]:
model.eval()

predictions = []
filenames = []

with torch.no_grad():
    for images, names in test_loader:
        images = images.to(device)

        outputs = model(images)

        # 고양이(Class 1)일 확률만 추출
        probs = torch.softmax(outputs, dim=1)[:, 1]

        predictions.extend(probs.cpu().numpy())
        filenames.extend(names)

In [44]:
submission = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")
submission["label"] = predictions

submission.to_csv(f"{DATA_DIR}/submission.csv", index=False)

In [45]:
submission

,id,label
0,1,9.998977e-01
1,2,5.042971e-09
2,3,1.579833e-01
3,4,9.999989e-01
4,5,9.726329e-01
...,...,...
12495,12496,9.539996e-01
12496,12497,9.710811e-01
12497,12498,9.985575e-01
12498,12499,6.507450e-06
